# Cookie Clicker Automater

Selenium notebook that opens [cookie-clicker2.com](https://cookie-clicker2.com/), clicks the big cookie, and buys the first four buildings (Cursor, Grandma, Farm, Mine) when you can afford them.

**Needs:** Python 3, `selenium`, and Google Chrome.

## Install (if needed)

```bash
pip install selenium
```

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## Open Cookie Clicker

Opens the site and hides common ad iframes so they do not steal clicks from the game.

In [ ]:
options = Options()
options.add_argument("--window-size=1400,1000")
options.add_argument("--disable-notifications")
# Uncomment if Chrome complains in a restricted environment:
# options.add_argument("--no-sandbox")
# options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 25)

driver.get("https://cookie-clicker2.com/")

# Ad iframes on this mirror often sit on top of the game and intercept Selenium clicks.
driver.execute_script(
    """
    document.querySelectorAll(
      'iframe[id^="aswift"], iframe[title="Advertisement"], #pxcelframe'
    ).forEach((el) => { el.style.display = 'none'; });
    """
)
print("Browser open. If a consent popup appears, accept it, then run Setup.")

## Setup

The game is **not** in the first ad iframe. Switch into `#iframehtml5`, then its nested game iframe, and grab the cookie + building buttons.

In [ ]:
def js_click(element):
    """Click via JavaScript so ad overlays are less likely to block the action."""
    driver.execute_script("arguments[0].click();", element)


driver.switch_to.default_content()

# Optional consent / language buttons on the outer page or in-game.
for selector in [
    "button.fc-cta-consent",
    "button.fc-button.fc-cta-consent",
    "#langSelect-EN",
]:
    for el in driver.find_elements(By.CSS_SELECTOR, selector):
        try:
            js_click(el)
        except Exception:
            pass

game_frame = wait.until(EC.presence_of_element_located((By.ID, "iframehtml5")))
driver.switch_to.frame(game_frame)

nested = wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, "iframe")))
if not nested:
    raise RuntimeError("No nested game iframe inside #iframehtml5. Refresh and re-run Setup.")
driver.switch_to.frame(nested[0])

# Language picker can appear inside the game frame.
for el in driver.find_elements(By.CSS_SELECTOR, "#langSelect-EN"):
    js_click(el)

clickplace_cursor = wait.until(EC.presence_of_element_located((By.ID, "product0")))
clickplace_grandma = driver.find_element(By.ID, "product1")
clickplace_farm = driver.find_element(By.ID, "product2")
clickplace_mine = driver.find_element(By.ID, "product3")
wait.until(EC.presence_of_element_located((By.ID, "bigCookie")))

print("Setup complete:", driver.find_element(By.ID, "cookies").text.replace("\n", " | "))

## Start Automation

Clicks the cookie in a loop and buys buildings 0–3 when affordable.

Stop with **Kernel → Interrupt**.

In [ ]:
LOCKED_PRICE = 10**30  # locked / blank / non-numeric prices = unaffordable


def parse_cookie_count(text):
    raw = text.split(" ")[0].replace(",", "").strip()
    return int(float(raw or "0"))


def parse_price(text):
    raw = text.replace(",", "").strip()
    if not raw:
        return LOCKED_PRICE
    try:
        return float(raw)
    except ValueError:
        return LOCKED_PRICE


buildings = [
    ("productPrice0", clickplace_cursor),
    ("productPrice1", clickplace_grandma),
    ("productPrice2", clickplace_farm),
    ("productPrice3", clickplace_mine),
]

try:
    while True:
        js_click(driver.find_element(By.ID, "bigCookie"))
        cookies = parse_cookie_count(driver.find_element(By.ID, "cookies").text)

        for price_id, button in buildings:
            price = parse_price(driver.find_element(By.ID, price_id).text)
            if cookies >= price:
                js_click(button)
                cookies = parse_cookie_count(driver.find_element(By.ID, "cookies").text)
except KeyboardInterrupt:
    print("Stopped automation.")
    print("Final:", driver.find_element(By.ID, "cookies").text.replace("\n", " | "))